#  Text-to-Speech (TTS) system Fun-CosyVoice 3.0 and OpenVINO

Fun-CosyVoice 3.0 is an advanced text-to-speech (TTS) system based on large language models (LLM), surpassing its predecessor (CosyVoice 2.0) in content consistency, speaker similarity, and prosody naturalness. It is designed for zero-shot multilingual speech synthesis in the wild.

More details can be found in the original [repository](https://github.com/FunAudioLLM/CosyVoice.git) and [model card](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512)

In this tutorial we consider how to run and optimize Fun-CosyVoice 3.0 using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Speech Generation](#Run-Speech-Generation)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/janus-multimodal-generation/janus-multimodal-generation.ipynb" />


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/fireredtts2/fireredtts2.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("fireredtts2.ipynb")

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform
import shutil

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "nncf",
    "openvino>=2025.4.0",
    "gradio",
    "huggingface_hub",
)

repo_dir = Path("CosyVoice")
revision = "8b54619760fcb78abf5e4637a88e19c1b9ab53c9"
clone_repo("https://github.com/FunAudioLLM/CosyVoice.git", revision)

# Replace CosyVoice requirements.txt with local version
shutil.copy("requirements.txt", repo_dir / "requirements.txt")

pip_install(
    "-q -r",
    str(repo_dir / "requirements.txt"),
)
if platform.system() == "Darwin":
    pip_install("numpy<2.0")

Found existing installation: fireredtts2 0.1
Uninstalling fireredtts2-0.1:
  Successfully uninstalled fireredtts2-0.1



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
  DEPRECATION: Legacy editable install of fireredtts2==0.1 from file:///home/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2 (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)


In [ ]:
from ov_cosyvoice_helper import convert_cosyvoice
from huggingface_hub import snapshot_download

pt_model_path = "pretrained_models"
snapshot_download(repo_id="FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=Path(pt_model_path))

model_path = "Fun-CosyVoice3-0.5B-2512-ov"
convert_cosyvoice(pt_model_path, model_path)

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✅ pretrained_models model already converted. You can find results in FireRedTTS2-ov


PosixPath('FireRedTTS2-ov')

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

llm_device = device_widget("CPU", exclude=["AUTO"])

llm_device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [ ]:
npu_ov_config = {
            "NPU_USE_NPUW" : "YES",
            "NPUW_LLM" : "YES",
            "NPUW_LLM_SHARED_HEAD" : "False",
            "NPUW_ONLINE_PIPELINE" : "NONE",
        }

In [ ]:
import sys
sys.path.append('CosyVoice/third_party/Matcha-TTS')
from ov_cosyvoice_helper import OVCosyVoice3


# Initialize OpenVINO CosyVoice3
# model_dir: OpenVINO model directory containing all converted models and dependency files
# If dependency files are not in ov_model_dir, provide original model_dir as second argument
cosyvoice = OVCosyVoice3(
    model_dir=model_path,
    device='CPU',  # Default device for all models
    llm_device=llm_device.value,      # LLM model device (defaults to device)
    flow_device='CPU',     # Flow model device (defaults to device)
    hift_device='CPU',     # HiFT model device (defaults to device)
    frontend_device='CPU', # Frontend model device (defaults to device)
    npu_ov_config=npu_ov_config,
)

[INFO] OV model Loaded...
[INFO] Text Tokenizer Loaded...


To deploy LLM on NPU, you can try

### Run zero_shot inference
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import torchaudio
import IPython


for i, j in enumerate(cosyvoice.inference_zero_shot(
    '八百标兵奔北坡，北坡炮兵并排跑，炮兵怕把标兵碰，标兵怕碰炮兵炮。',
    'You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。',
    './asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_zero_shot_{i}.wav")
    display(IPython.display.Audio(f"ov_zero_shot_{i}.wav"))


### Run cross_lingual inference with fine-grained control
[back to top ⬆️](#Table-of-contents:)

In [ ]:
for i, j in enumerate(cosyvoice.inference_cross_lingual(
    'You are a helpful assistant.<|endofprompt|>[breath]因为他们那一辈人[breath]在乡里面住的要习惯一点，[breath]邻居都很活络，[breath]嗯，都很熟悉。[breath]',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_fine_grained_control_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_fine_grained_control_{i}.wav")
    display(IPython.display.Audio(f"ov_fine_grained_control_{i}.wav"))

### Run instruct2 inference (Cantonese)
[back to top ⬆️](#Table-of-contents:)

In [ ]:
for i, j in enumerate(cosyvoice.inference_instruct2(
    '好少咯，一般系放嗰啲国庆啊，中秋嗰啲可能会咯。',
    'You are a helpful assistant. 请用广东话表达。<|endofprompt|>',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_instruct_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_instruct_{i}.wav")
    display(IPython.display.Audio(f"ov_instruct_{i}.wav"))

### Run instruct2 inference (fast speed)
[back to top ⬆️](#Table-of-contents:)

In [ ]:
for i, j in enumerate(cosyvoice.inference_instruct2(
    '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。',
    'You are a helpful assistant. 请用尽可能快地语速说一句话。<|endofprompt|>',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_instruct_fast_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_instruct_fast_{i}.wav")
    display(IPython.display.Audio(f"ov_instruct_fast_{i}.wav"))

### Run hotfix inference (fast speed)
[back to top ⬆️](#Table-of-contents:)

In [ ]:
for i, j in enumerate(cosyvoice.inference_zero_shot(
    '高管也通过电话、短信、微信等方式对报道[j][ǐ]予好评。',
    'You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_hotfix_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_hotfix_{i}.wav")
    display(IPython.display.Audio(f"ov_hotfix_{i}.wav"))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)